In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

try:
    from adjustText import adjust_text
except ModuleNotFoundError:

    def adjust_text(*args, **kwargs):
        return None


import matplotlib.patheffects as patheffects

MODEL_ORDER = ["GAT", "2-Hop GAT", "Filtered 2-Hop GAT"]
# Subsumption is excluded from Pizza 100 plots because the fixed val/test files
# contain Family/Genealogy subclass triples rather than Pizza subclass triples.
# TASKS = ["Membership", "Subsumption", "Link Prediction"]
TASKS = ["Membership", "Link Prediction"]
RESULTS_DIR = "results"


def parse_result_filename(filename, ontology_name):
    stem = (
        filename.removesuffix(".txt")
        .removesuffix("_nsorn_protocol")
        .removesuffix("_test")
    )
    if stem == ontology_name:
        return ontology_name, "None", 0

    prefix = f"{ontology_name}_"
    if not stem.startswith(prefix):
        raise ValueError(f"Unexpected filename for {ontology_name}: {filename}")

    noise_part = stem[len(prefix) :]
    noise_type, noise_level = noise_part.rsplit("_", 1)
    return ontology_name, noise_type, noise_level


def get_results_mean(ontology_name, metrics_index=0, results_dir=RESULTS_DIR):
    file_pattern = os.path.join(
        results_dir, f"{ontology_name}*_test_nsorn_protocol.txt"
    )
    data = []

    for filepath in sorted(glob.glob(file_pattern)):
        filename = os.path.basename(filepath)
        try:
            ontology, noise_type, noise_level = parse_result_filename(
                filename, ontology_name
            )
        except ValueError as exc:
            print(exc)
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()

        current_model = None
        current_task = None
        for line in lines:
            line = line.strip()
            if line.startswith("Model:"):
                current_model = line.replace("Model:", "").strip()
                current_task = None
            elif line.rstrip(":") in TASKS:
                current_task = line.rstrip(":")
            elif line.startswith("Mean:") and current_model and current_task:
                mean_values = line.replace("Mean:", "").split("&")
                mean_values = [float(x.strip()) for x in mean_values if x.strip()]
                if len(mean_values) > metrics_index:
                    data.append(
                        {
                            "Ontology": ontology,
                            "Reasoner": current_model,
                            "NoiseType": noise_type,
                            "NoiseLevel": noise_level,
                            "Task": current_task,
                            "MRR": mean_values[metrics_index],
                        }
                    )
                current_task = None

    return pd.DataFrame(data)


def visualize(df, noise_type, title, output_file_name):
    sns.set_theme(style="whitegrid", font_scale=1.3)

    df_noise_type = df[df["NoiseType"] == noise_type].copy()
    df_noise_type["NoiseLevel"] = pd.to_numeric(
        df_noise_type["NoiseLevel"], errors="coerce"
    )
    df_noise_type["MRR"] = pd.to_numeric(df_noise_type["MRR"], errors="coerce")
    df_noise_type = df_noise_type.dropna(subset=["NoiseLevel", "MRR"])

    model_dfs = [
        df_noise_type[df_noise_type["Reasoner"] == model] for model in MODEL_ORDER
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    def plot_line_with_values(ax, data, title):
        ax.set_title(title)
        ax.set_ylim(0, 1.1)
        ax.set_xlabel("Noise Level")
        ax.set_ylabel("MRR" if ax == axes[0] else "")
        ax.grid(True)

        if data.empty:
            return

        palette = sns.color_palette("Set2", n_colors=data["Task"].nunique())
        task_order = data["Task"].drop_duplicates().tolist()
        task_color_map = {task: palette[i] for i, task in enumerate(task_order)}

        sns.lineplot(
            data=data,
            x="NoiseLevel",
            y="MRR",
            hue="Task",
            marker="o",
            palette="Set2",
            err_style="bars",
            errorbar="sd",
            ax=ax,
        )

        min_x, max_x = data["NoiseLevel"].min(), data["NoiseLevel"].max()
        ticks = list(np.arange(min_x, max_x + 0.25, 0.25))
        ax.set_xticks(ticks)

        texts = []
        for task in data["Task"].unique():
            task_data = data[data["Task"] == task]
            label_color = task_color_map.get(task, "black")
            for _, row in task_data.iterrows():
                texts.append(
                    ax.text(
                        row["NoiseLevel"],
                        row["MRR"] + 0.04,
                        f"{row['MRR']:.3f}",
                        color=label_color,
                        ha="center",
                        va="bottom",
                        fontsize=12,
                        fontweight="bold",
                        path_effects=[
                            patheffects.withStroke(linewidth=3, foreground="white")
                        ],
                    )
                )
        adjust_text(texts, ax=ax)

    for ax, model_df, model_name in zip(axes, model_dfs, MODEL_ORDER):
        plot_line_with_values(ax, model_df, model_name)

    handles, labels = axes[2].get_legend_handles_labels()
    if handles:
        axes[2].legend(handles, labels, title="Task", loc="upper right")
    for ax in axes[:2]:
        if ax.legend_:
            ax.legend_.remove()

    save_dir = os.path.join(RESULTS_DIR, "figures")
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, output_file_name)
    fig.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(save_path, dpi=300)


def main(ontology_name, noise_type):
    df = get_results_mean(ontology_name, metrics_index=0)
    if df.empty:
        raise FileNotFoundError(f"No NSORN result files found for {ontology_name}")

    noise_types = ["random", "logical", "gnn"]
    dfs = [
        df[df["NoiseType"] == "None"].assign(NoiseType=noise) for noise in noise_types
    ]
    df_no_noise = pd.concat(dfs, ignore_index=True)
    df = pd.concat([df, df_no_noise], ignore_index=True)

    display_name = ontology_name.replace("noise_", "").replace("_", " ").title()
    visualize(
        df,
        noise_type,
        f"{noise_type.capitalize()} Noise - {display_name}",
        f"{ontology_name}_{noise_type}.png",
    )

In [ ]:
main("noise_pizza_100_pizza_100", "random")
main("noise_pizza_100_pizza_100", "gnn")
main("noise_pizza_100_pizza_100", "logical")

In [ ]:
def get_results_mean_all_metrics(
    ontology_name, output_file=None, results_dir=RESULTS_DIR
):
    file_pattern = os.path.join(
        results_dir, f"{ontology_name}*_test_nsorn_protocol.txt"
    )
    data = []

    for filepath in sorted(glob.glob(file_pattern)):
        filename = os.path.basename(filepath)
        try:
            ontology, noise_type, noise_level = parse_result_filename(
                filename, ontology_name
            )
        except ValueError as exc:
            print(exc)
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()

        current_model = None
        current_task = None
        for line in lines:
            line = line.strip()
            if line.startswith("Model:"):
                current_model = line.replace("Model:", "").strip()
                current_task = None
            elif line.rstrip(":") in TASKS:
                current_task = line.rstrip(":")
            elif line.startswith("Mean:") and current_model and current_task:
                mean_values = line.replace("Mean:", "").split("&")
                mean_values = [float(x.strip()) for x in mean_values if x.strip()]
                data.append(
                    {
                        "Ontology": ontology,
                        "Reasoner": current_model,
                        "NoiseType": noise_type,
                        "NoiseLevel": noise_level,
                        "Task": current_task,
                        "Metrics": mean_values,
                    }
                )
                current_task = None

    if output_file and data:
        max_metrics_len = max(len(entry["Metrics"]) for entry in data)
        headers = ["Ontology", "Reasoner", "NoiseType", "NoiseLevel", "Task"] + [
            f"Metric_{i + 1}" for i in range(max_metrics_len)
        ]
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as out_f:
            out_f.write("\t".join(headers) + "\n")
            for entry in data:
                metrics = entry["Metrics"] + [""] * (
                    max_metrics_len - len(entry["Metrics"])
                )
                row = [
                    entry["Ontology"],
                    entry["Reasoner"],
                    entry["NoiseType"],
                    str(entry["NoiseLevel"]),
                    entry["Task"],
                ] + [str(m) for m in metrics]
                out_f.write("\t".join(row) + "\n")

    return pd.DataFrame(data)

In [ ]:
df_noise_pizza_100 = get_results_mean_all_metrics(
    "noise_pizza_100_pizza_100",
    output_file="results/noise_pizza_100_pizza_100_gat_variants.txt",
)